# 📊 checkSIGNALs — バックテスト用データ書出しツール

**出力形式**: `output/XXXXX.csv` （1銘柄×全日付＝1ファイル）  
**ファイル名例**: `7203T.csv`（日本株）/ `AAPL_.csv`（米国株）  
**出力列**: 日付 / 四半期 / ティッカー / 銘柄名 / 株価 / Q / V / T / QVTスコア  
**欠損日**: `-` で埋めてスキップ（プログラムは止まらない）  

---
### 使い方
1. **セル1（設定）** の `TICKERS` と `DATES_CSV` を編集
2. 「すべて実行」で完了


In [2]:
# ═══════════════════════════════════════════════════════════════
# セル1: 設定（ここだけ編集する）
# ═══════════════════════════════════════════════════════════════

# ▼ 銘柄リスト（最大30銘柄）
TICKERS = [
    "7203.T",  # トヨタ
    "3864.T",  # 三菱製紙
    "8113.T",  # ユニ・チャーム
    "7932.T",  # ニッピ
    "5333.T",  # 日本ガイシ
    # --- 追加はここに ---
]

# ▼ 日付リストCSVのパス
# フォーマット: 1列（ヘッダーなし）、YYYYMMDD形式の整数 or 文字列
#   20250106
#   20250107
#   20250108
#   ...
DATES_CSV = "dates.csv"   # ← ファイルパスを変更する場合はここだけ

import pandas as pd
from datetime import date

_df_dates = pd.read_csv(DATES_CSV, header=None, dtype=str)
_raw      = _df_dates.iloc[:, 0].dropna().str.strip().tolist()

# YYYYMMDD → YYYY-MM-DD に変換 & ソート
TARGET_DATES = sorted(
    f"{d[:4]}-{d[4:6]}-{d[6:8]}"
    for d in _raw
    if len(d) == 8 and d.isdigit()
)

if not TARGET_DATES:
    raise ValueError(f"{DATES_CSV} から有効な日付（YYYYMMDD形式）が読み込めませんでした")

# ▼ CSV出力先フォルダ
OUTPUT_DIR = "output"

# ▼ 既存CSVを上書きするか（False = スキップ）
OVERWRITE = True

# ──────────────────────────────────────────
# ▼ J-Quants 設定
# ──────────────────────────────────────────
# メールアドレスとパスワードを設定するとリフレッシュトークンを自動取得する。
# 空文字列 "" のままにすると yfinance（現在値）にフォールバック。
JQUANTS_MAILADDRESS = "the3stigmataofpalmereldritch@gmail.com"   # 例: "your@email.com"
JQUANTS_PASSWORD    = "469Maonethousand"   # 例: "your_password"

# キャッシュを再生成するか（False = 既存キャッシュを再利用・推奨）
JQUANTS_REFRESH_CACHE = false

print(f"銘柄数  : {len(TICKERS)}")
print(f"対象日数: {len(TARGET_DATES)} 日")
print(f"期間    : {TARGET_DATES[0]} → {TARGET_DATES[-1]}")
print(f"出力先  : {OUTPUT_DIR}/")

銘柄数  : 5
対象日数: 104 日
期間    : 2023-10-01 → 2025-02-27
出力先  : output/


In [4]:
# ═══════════════════════════════════════════════════════════════
# セル1.5: J-Quantsキャッシュ生成
#   メアド・パスワード → リフレッシュトークン自動取得 → キャッシュ生成
#   JQUANTS_REFRESH_CACHE = False かつ キャッシュ既存の場合はスキップ
# ═══════════════════════════════════════════════════════════════

import requests, json
from pathlib import Path

_cache_path = Path(OUTPUT_DIR) / 'fundamentals_cache.csv'

if not JQUANTS_MAILADDRESS or not JQUANTS_PASSWORD:
    print('⚠️  JQUANTS_MAILADDRESS / JQUANTS_PASSWORD が未設定 → yfinance にフォールバック')
    JQUANTS_REFRESH_TOKEN = ''

elif not JQUANTS_REFRESH_CACHE and _cache_path.exists():
    print(f'✅ キャッシュ既存のためスキップ: {_cache_path}')
    print('   再生成する場合は JQUANTS_REFRESH_CACHE = True に変更してください')
    JQUANTS_REFRESH_TOKEN = '(cached)'

else:
    # メアド・パスワード → リフレッシュトークンを取得
    print('J-Quants: リフレッシュトークン取得中...')
    _resp = requests.post(
        'https://api.jquants.com/v1/token/auth_user',
        data=json.dumps({'mailaddress': JQUANTS_MAILADDRESS, 'password': JQUANTS_PASSWORD})
    )
    if _resp.status_code != 200:
        raise ValueError(f'リフレッシュトークン取得失敗: {_resp.status_code} {_resp.text}')

    JQUANTS_REFRESH_TOKEN = _resp.json().get('refreshToken', '')
    if not JQUANTS_REFRESH_TOKEN:
        raise ValueError(f'refreshToken が取得できませんでした: {_resp.json()}')
    print(f'✅ リフレッシュトークン取得OK')

    # キャッシュ生成
    print('キャッシュ生成中（約6分）...')
    %run ../02_データベースbuild/fetch_jquants_cache.py


✅ キャッシュ既存のためスキップ: output\fundamentals_cache.csv
   再生成する場合は JQUANTS_REFRESH_CACHE = True に変更してください


In [5]:
# ═══════════════════════════════════════════════════════════════
# セル2: インポート・モジュール読み込み
# ═══════════════════════════════════════════════════════════════
import os, sys, time, warnings
import traceback
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Any

import pandas as pd
import numpy as np
import yfinance as yf

warnings.filterwarnings("ignore")

# app/modules/ をパスに追加（checkSIGNALs のプロジェクト構造）
# ├── backtest_export.ipynb  ← このファイル
# └── app/
#     └── modules/
#         ├── data_fetch.py
#         ├── indicators.py
#         ├── q_logic.py
#         ├── valuation.py
#         ├── t_logic.py
#         ├── q_correction.py
#         ├── pattern_db.py
#         └── __init__.py

NOTEBOOK_DIR = Path().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR / "app"))

from modules.q_logic    import score_quality
from modules.valuation  import score_valuation
from modules.t_logic    import compute_t_metrics
from modules.indicators import (
    calc_moving_averages, calc_bollinger_bands, calc_rsi, calc_slope
)

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print("✅ インポート完了")

✅ インポート完了


In [6]:
# ═══════════════════════════════════════════════════════════════
# セル3: ユーティリティ関数
# ═══════════════════════════════════════════════════════════════

NA = "-"  # 欠損値の文字列表現

def _safe(x, digits=2):
    """float を丸めて返す。None / NaN は '-' にする。"""
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return NA
        return round(float(x), digits)
    except Exception:
        return NA


def fetch_ohlcv(ticker: str, end_date_str: str, lookback_days: int = 300) -> Optional[pd.DataFrame]:
    """
    指定日を「最終取引日」として lookback_days 分の OHLCV を取得。
    yfinance の end は翌日を指定する必要あり。
    データ取得失敗 / 空のときは None を返す。
    """
    end_dt   = datetime.strptime(end_date_str, "%Y-%m-%d") + timedelta(days=1)
    start_dt = end_dt - timedelta(days=lookback_days)

    for attempt in range(2):
        try:
            df = yf.download(
                ticker,
                start=start_dt.strftime("%Y-%m-%d"),
                end=end_dt.strftime("%Y-%m-%d"),
                interval="1d",
                progress=False,
                auto_adjust=True,
            )
            if df is not None and not df.empty:
                # MultiIndex 対応
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = ["_".join(c).strip() for c in df.columns]
                return df
        except Exception:
            pass
        time.sleep(0.5)
    return None


def get_close_col(df: pd.DataFrame) -> Optional[str]:
    # カラム名はすでに小文字統一済み（"close" を直接探す）
    for c in df.columns:
        if str(c).lower().strip() == "close":
            return c
    # フォールバック: "close" を含むカラム
    for c in df.columns:
        if "close" in str(c).lower():
            return c
    return None


def make_csv_filename(ticker: str) -> str:
    """
    XXXXX.csv 形式のファイル名を生成する（銘柄ごと全日付を1ファイルに格納）。
    XXXXX = ティッカー部分（5桁固定）

    日本株: 7203.T  → '7203T.csv'
    米国株: AAPL    → 'AAPL_.csv'  （4文字は _ で補完）
            GOOGL   → 'GOOGL.csv'
            BRK.B   → 'BRKB_.csv'  （ピリオド除去 + _ 補完）
    """
    t = ticker.strip().upper()
    is_jpx = t.endswith(".T") or (t.isdigit() and 4 <= len(t) <= 5)
    if is_jpx:
        code = t.replace(".T", "")[:4]
        ticker_part = f"{code}T"
    else:
        clean = t.replace(".", "").replace("-", "")
        ticker_part = clean[:5].ljust(5, '_')
    return f"{ticker_part}.csv"


def get_fundamentals(ticker: str) -> Dict[str, Any]:
    """
    yfinance.Ticker.info からファンダメンタル指標を取得。
    ※ファンダメンタルは「現時点」の値（過去日付への対応はAPIの制約により困難）
    """
    result = {k: None for k in [
        "company_name", "eps", "bps", "per_fwd",
        "roe", "roa", "equity_ratio",
        "operating_margin", "de_ratio", "interest_coverage",
        "ev_ebitda", "dividend_yield",
        "industry", "sector",
    ]}
    try:
        t    = yf.Ticker(ticker)
        info = t.info or {}

        def _f(key, scale=1.0):
            v = info.get(key)
            try: return float(v) * scale if v not in (None, "", "None") else None
            except: return None

        result["company_name"]      = info.get("longName") or info.get("shortName") or ticker
        result["eps"]               = _f("trailingEps")
        result["bps"]               = _f("bookValue")
        result["per_fwd"]           = _f("forwardPE")
        result["roe"]               = _f("returnOnEquity",  100.0)
        result["roa"]               = _f("returnOnAssets",  100.0)
        result["operating_margin"]  = _f("operatingMargins", 100.0)
        result["de_ratio"]          = _f("debtToEquity", 1/100.0) if info.get("debtToEquity") else None
        result["ev_ebitda"]         = _f("enterpriseToEbitda")
        result["industry"]          = info.get("industry", "")
        result["sector"]            = info.get("sector", "")

        # 配当利回り
        divs = t.dividends
        if isinstance(divs, pd.Series) and len(divs) > 0:
            divs.index = pd.to_datetime(divs.index, errors="coerce")
            try:
                if getattr(divs.index, "tz", None):
                    divs.index = divs.index.tz_localize(None)
            except Exception:
                pass
            one_year_ago = datetime.now() - timedelta(days=365)
            # 配当利回り計算は後で close が必要なので後回し（Noneのまま）
            result["_dividends_1y"] = float(divs[divs.index >= one_year_ago].sum())
        else:
            result["_dividends_1y"] = 0.0

        # equity_ratio: D/Eから逆算 or financials
        roe = result["roe"]
        roa = result["roa"]
        if roe and roa and roe != 0:
            approx = (roa / 100) / (roe / 100)
            if 0 < approx < 1:
                result["equity_ratio"] = approx * 100

        # interest_coverage: 経路① ebit / interestExpense (info直接)
        ebit   = _f("ebit")
        int_ex = _f("interestExpense")
        if ebit and int_ex and int_ex != 0:
            result["interest_coverage"] = round(abs(ebit / int_ex), 2)

        # interest_coverage: 経路② financials DataFrame からフォールバック
        if result["interest_coverage"] is None:
            try:
                fin = t.financials
                if fin is not None and not fin.empty:
                    col = fin.columns[0]
                    def _get_fin(keys):
                        for k in keys:
                            for idx in fin.index:
                                if k.lower() in str(idx).lower():
                                    v = fin.loc[idx, col]
                                    if v is not None and str(v) not in ("nan", "None", "NaN"):
                                        return float(v)
                        return None
                    ebit_fin  = _get_fin(["EBIT", "Operating Income"])
                    intex_fin = _get_fin(["Interest Expense", "Interest And Debt Expense"])
                    if ebit_fin and intex_fin and intex_fin != 0:
                        result["interest_coverage"] = round(abs(ebit_fin / intex_fin), 2)
            except Exception:
                pass

    except Exception as e:
        pass
    return result


def calc_scores_for_date(
    df_full: pd.DataFrame,
    close_col: str,
    target_date_str: str,
    fundamentals: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    """
    df_full（過去300日分のOHLCV）から target_date 時点のスコアを計算。
    対象日のデータが存在しなければ None を返す。
    """
    target_dt = pd.Timestamp(target_date_str)

    # ─ 対象日までのデータを切り出す ─
    df_full.index = pd.to_datetime(df_full.index)
    try:
        if df_full.index.tz is not None:
            df_full.index = df_full.index.tz_localize(None)
    except Exception:
        pass

    df_slice = df_full[df_full.index <= target_dt].copy()

    if df_slice.empty or len(df_slice) < 25:
        return None  # データ不足

    # ─ 対象日の終値が存在するか確認 ─
    last_idx  = df_slice.index[-1]
    last_date = last_idx.date()
    target_d  = pd.Timestamp(target_date_str).date()

    # 当日データがない（休場など）は None
    if last_date != target_d:
        return None

    # ─ テクニカル指標計算 ─
    df_slice = calc_moving_averages(df_slice, close_col)
    df_slice = calc_bollinger_bands(df_slice, close_col)
    df_slice = calc_rsi(df_slice, close_col)

    df_valid = df_slice.dropna(subset=[close_col, "25MA", "50MA", "75MA", "RSI"])
    if df_valid.empty:
        return None

    last = df_valid.iloc[-1]
    price    = float(last[close_col])
    ma_25    = float(last["25MA"])
    ma_50    = float(last["50MA"])
    ma_75    = float(last["75MA"])
    rsi      = float(last["RSI"])
    bb_p1    = float(last["BB_+1σ"])
    bb_p2    = float(last["BB_+2σ"])
    bb_m1    = float(last["BB_-1σ"])
    bb_m2    = float(last["BB_-2σ"])
    slope_25 = calc_slope(df_slice["25MA"])

    # 52週高値/安値
    df_1y = df_slice.iloc[-252:]
    high_col = next((c for c in df_slice.columns if c.lower().startswith("high")), None)
    low_col  = next((c for c in df_slice.columns if c.lower().startswith("low")),  None)
    high_52w = float(df_1y[high_col].max()) if high_col else float(df_1y[close_col].max())
    low_52w  = float(df_1y[low_col].min())  if low_col  else float(df_1y[close_col].min())

    # PER / PBR
    eps = fundamentals.get("eps")
    bps = fundamentals.get("bps")
    per = (price / eps) if eps and eps != 0 else None
    pbr = (price / bps) if bps and bps != 0 else None

    # 配当利回り
    divs_1y = fundamentals.get("_dividends_1y", 0.0) or 0.0
    dividend_yield = (divs_1y / price * 100.0) if price > 0 and divs_1y > 0 else None

    industry = fundamentals.get("industry") or ""
    sector   = fundamentals.get("sector")   or ""
    is_us    = not (ticker.endswith(".T") or ticker.replace(".T","").isdigit())

    # ─ Q スコア ─
    q_result = score_quality(
        roe=fundamentals.get("roe"),
        roa=fundamentals.get("roa"),
        equity_ratio=fundamentals.get("equity_ratio"),
        operating_margin=fundamentals.get("operating_margin"),
        de_ratio=fundamentals.get("de_ratio"),
        interest_coverage=fundamentals.get("interest_coverage"),
        industry=industry,
        sector=sector,
        is_us=is_us,
    )
    q_score = q_result["q_score"]
    q1      = q_result["q1"]
    q3      = q_result["q3"]

    # ─ V スコア ─
    v_result = score_valuation(
        per=per, pbr=pbr,
        dividend_yield=dividend_yield,
        ev_ebitda=fundamentals.get("ev_ebitda"),
        is_us=is_us,
    )
    v_score = v_result["v_score"]
    v1, v2, v3 = v_result["v1"], v_result["v2"], v_result["v3"]

    # ─ T スコア ─
    t_metrics = compute_t_metrics(
        price=price, ma_25=ma_25, ma_50=ma_50, ma_75=ma_75,
        rsi=rsi,
        bb_plus1=bb_p1, bb_plus2=bb_p2,
        bb_minus1=bb_m1, bb_minus2=bb_m2,
        slope_25=slope_25,
        low_52w=low_52w, high_52w=high_52w,
        per=per, pbr=pbr,
    )
    t_score = t_metrics["t_score"]

    # ─ QVT 総合 ─
    qvt_score = round((q_score + v_score + t_score) / 3.0, 1)

    return {
        "close":   price,
        "ma_25":   ma_25,
        "ma_50":   ma_50,
        "rsi":     rsi,
        "high_52w": high_52w,
        "low_52w":  low_52w,
        "per": per, "pbr": pbr,
        "q_score": q_score, "q1": q1, "q3": q3,
        "v_score": v_score, "v1": v1, "v2": v2, "v3": v3,
        "t_score": t_score,
        "qvt_score": qvt_score,
        "timing_label": t_metrics.get("timing_label", ""),
        "signal_text":  t_metrics.get("signal_text",  ""),
    }


print("✅ 関数定義完了")

✅ 関数定義完了


In [7]:
# ═══════════════════════════════════════════════════════════════
# セル4: 四半期キャッシュ管理
#   fundamentals_cache.csv（fetch_jquants_cache.py が生成）を読み込み、
#   サンプリング日ごとに「その四半期末以前で最新の決算値」を返す。
#   キャッシュがない場合は yfinance（現在値）にフォールバック。
# ═══════════════════════════════════════════════════════════════

def get_quarter_key(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    q  = (dt.month - 1) // 3 + 1
    return f"{dt.year}-Q{q}"


# ── キャッシュCSV読み込み ──────────────────────────────────────
CACHE_PATH = Path(OUTPUT_DIR) / "fundamentals_cache.csv"
_fund_df: Optional[pd.DataFrame] = None

if CACHE_PATH.exists():
    _fund_df = pd.read_csv(CACHE_PATH, encoding="utf-8-sig", dtype=str)
    _num_cols = ["roe","roa","operating_margin","equity_ratio",
                 "de_ratio","interest_coverage","eps","bps","ev_ebitda"]
    for col in _num_cols:
        if col in _fund_df.columns:
            _fund_df[col] = pd.to_numeric(_fund_df[col], errors="coerce")
    DATA_SOURCE = "J-Quants（過去実績値）"
    print(f"✅ キャッシュ読み込み: {len(_fund_df)}行 / {_fund_df['ticker'].nunique()}銘柄")
else:
    DATA_SOURCE = "yfinance（現在値・近似）"
    print(f"⚠️  {CACHE_PATH} なし → yfinance にフォールバック")

print(f"データソース: {DATA_SOURCE}")


def _lookup(ticker: str, quarter_key: str) -> Optional[Dict[str, Any]]:
    """
    キャッシュから指定銘柄・四半期に最も近い決算データを返す。
    同一quarter_key内で disclosed_date が最新のものを優先。
    見つからない場合は直前四半期に遡る。
    """
    if _fund_df is None:
        return None
    df = _fund_df
    hit = df[(df["ticker"] == ticker) & (df["quarter_key"] == quarter_key)]
    if hit.empty:
        # 直前四半期にフォールバック
        candidates = df[(df["ticker"] == ticker) & (df["quarter_key"] <= quarter_key)]
        if candidates.empty:
            return None
        hit = candidates.sort_values("quarter_key", ascending=False).head(1)
    else:
        hit = hit.sort_values("disclosed_date", ascending=False).head(1)

    row = hit.iloc[0]
    return {k: (None if pd.isna(v) else v)
            for k, v in row[["roe","roa","operating_margin","equity_ratio",
                              "de_ratio","interest_coverage","eps","bps","ev_ebitda"]].items()}


def fetch_fund_for_quarter(ticker: str, quarter_key: str) -> Dict[str, Any]:
    """
    優先順位:
      1. fundamentals_cache.csv（J-Quants過去実績値）
      2. yfinance（現在値・フォールバック）
    """
    result: Dict[str, Any] = {"_fetched_quarter": quarter_key}

    cached = _lookup(ticker, quarter_key)
    if cached is not None:
        result.update(cached)
        result["_source"] = "jquants_cache"
        try:
            info = yf.Ticker(ticker).info or {}
            result["company_name"]   = info.get("longName") or info.get("shortName") or ticker
            result["industry"]       = info.get("industry", "")
            result["sector"]         = info.get("sector",   "")
            result["dividend_yield"] = None
        except Exception:
            result.setdefault("company_name", ticker)
            result.setdefault("industry", "")
            result.setdefault("sector",   "")
        return result

    # yfinance フォールバック
    try:
        yf_data = get_fundamentals(ticker)
        yf_data["_fetched_quarter"] = quarter_key
        yf_data["_source"] = "yfinance"
        return yf_data
    except Exception as e:
        return {"_fetched_quarter": quarter_key, "_source": "yfinance", "_error": str(e)}


# ── 対象四半期一覧 ────────────────────────────────────────────
all_quarters = sorted(set(get_quarter_key(d) for d in TARGET_DATES))

print("\n対象四半期:")
for qk in all_quarters:
    dates_in_q = [d for d in TARGET_DATES if get_quarter_key(d) == qk]
    print(f"  {qk}: {dates_in_q[0]} 〜 {dates_in_q[-1]} ({len(dates_in_q)}日)")

print(f"\n→ ファンダメンタル取得: {len(TICKERS)}銘柄 × {len(all_quarters)}四半期 = {len(TICKERS)*len(all_quarters)}回")
print("✅ 四半期ユーティリティ定義完了")


✅ キャッシュ読み込み: 46行 / 5銘柄
データソース: J-Quants（過去実績値）

対象四半期:
  2023-Q4: 2023-10-01 〜 2023-10-31 (31日)
  2024-Q1: 2024-01-30 〜 2024-02-27 (21日)
  2024-Q4: 2024-10-01 〜 2024-10-31 (31日)
  2025-Q1: 2025-01-30 〜 2025-02-27 (21日)

→ ファンダメンタル取得: 5銘柄 × 4四半期 = 20回
✅ 四半期ユーティリティ定義完了


In [8]:
# ═══════════════════════════════════════════════════════════════
# セル5: メイン実行
# ═══════════════════════════════════════════════════════════════

from IPython.display import display

# ──────────────────────────────────────────────────────────────
# [1/3] ファンダメンタルを四半期×銘柄でまとめて取得
#   キー: (ticker, quarter_key)  例: ('7203.T', '2024-Q3')
# ──────────────────────────────────────────────────────────────
print("[1/3] ファンダメンタル取得中（四半期×銘柄）...")
print("      ※yfinanceは現在値のみ提供のため、四半期ごとに再フェッチして")
print("        バックテスト期間中の変化を近似的に反映します")
print()

# (ticker, quarter_key) → fundamentals dict
fund_cache: Dict[tuple, Dict] = {}

total_fetch = len(TICKERS) * len(all_quarters)
fetch_count = 0

for qk in all_quarters:
    print(f"  ▶ {qk}")
    for ticker in TICKERS:
        fetch_count += 1
        print(f"    [{fetch_count:3d}/{total_fetch}] {ticker} ...", end=" ")
        fund = fetch_fund_for_quarter(ticker, qk)
        fund_cache[(ticker, qk)] = fund
        name = fund.get("company_name") or ticker
        err  = fund.get("_error", "")
        print(name if not err else f"ERROR: {err}")
        time.sleep(0.25)   # API負荷軽減
    print()

# ──────────────────────────────────────────────────────────────
# [2/3] OHLCVを銘柄ごとに1回まとめて取得（全対象期間カバー）
#   最初の日付から252営業日分（≒1年）を先頭に余分に確保
# ──────────────────────────────────────────────────────────────
print("[2/3] 株価データ取得中...")
ohlcv_cache: Dict[str, Optional[pd.DataFrame]] = {}

latest_date  = TARGET_DATES[-1]
earliest_date = TARGET_DATES[0]

# 最古日付から252日前までを start にすることでMA/RSI計算に必要な行数を確保
earliest_dt    = datetime.strptime(earliest_date, "%Y-%m-%d")
ohlcv_start_dt = earliest_dt - timedelta(days=400)   # 400日＝MA75+余裕

for i, ticker in enumerate(TICKERS, 1):
    print(f"  {i:2d}/{len(TICKERS)} {ticker} ...", end=" ")
    try:
        end_dt = datetime.strptime(latest_date, "%Y-%m-%d") + timedelta(days=1)
        df_raw = yf.download(
            ticker,
            start=ohlcv_start_dt.strftime("%Y-%m-%d"),
            end=end_dt.strftime("%Y-%m-%d"),
            interval="1d",
            progress=False,
            auto_adjust=True,
        )
        if df_raw is not None and not df_raw.empty:
            # MultiIndex処理（yfinanceバージョン差異を吸収）
            if isinstance(df_raw.columns, pd.MultiIndex):
                lv0 = df_raw.columns.get_level_values(0).tolist()
                lv1 = df_raw.columns.get_level_values(1).tolist()
                # (Price, Ticker) 形式: lv1にtickerが入る
                if ticker in lv1:
                    df_raw.columns = [str(c).lower().strip() for c in lv0]
                # (Ticker, Price) 形式: lv0にtickerが入る
                elif ticker in lv0:
                    df_raw.columns = [str(c).lower().strip() for c in lv1]
                else:
                    df_raw.columns = ["_".join(str(x) for x in c).lower().strip()
                                      for c in df_raw.columns]
            else:
                df_raw.columns = [str(c).lower().strip() for c in df_raw.columns]
            # 空カラムを除去
            df_raw = df_raw.loc[:, df_raw.columns.str.strip() != ""]
            ohlcv_cache[ticker] = df_raw
            print(f"OK ({len(df_raw)} rows, {ohlcv_start_dt.strftime('%Y-%m-%d')} 〜 {latest_date})")
        else:
            ohlcv_cache[ticker] = None
            print("FAILED（空データ）")
    except Exception as e:
        ohlcv_cache[ticker] = None
        print(f"ERROR: {e}")
    time.sleep(0.3)

# ──────────────────────────────────────────────────────────────
# [3/3] 銘柄ループ → 全日付分のスコアを計算 → 銘柄ごと1CSV出力
#   ファイル名: XXXXX.csv（例: 7203T.csv / AAPL_.csv）
#   行数: 対象日付数
# ──────────────────────────────────────────────────────────────
print(f"\n[3/3] スコア計算 & CSV出力 ({len(TICKERS)}銘柄 × {len(TARGET_DATES)}日)...")

summary_log = []   # (ticker, ok_cnt, skip_cnt, err_cnt)

for ti, ticker in enumerate(TICKERS, 1):

    csv_name = make_csv_filename(ticker)
    csv_path = Path(OUTPUT_DIR) / csv_name

    if csv_path.exists() and not OVERWRITE:
        print(f"  [{ti:2d}/{len(TICKERS)}] {ticker}: スキップ（既存: {csv_name}）")
        continue

    df_raw    = ohlcv_cache.get(ticker)
    close_col = get_close_col(df_raw) if df_raw is not None else None

    rows         = []
    ok_cnt       = 0
    skip_cnt     = 0
    err_cnt      = 0
    prev_quarter = None

    for date_str in TARGET_DATES:

        quarter_key = get_quarter_key(date_str)
        if quarter_key != prev_quarter:
            prev_quarter = quarter_key

        fund = fund_cache.get((ticker, quarter_key), {})
        name = fund.get("company_name") or ticker

        row = {
            "date":         date_str,
            "quarter":      quarter_key,
            "ticker":       ticker,
            "company_name": name,
            "close":        NA, "ma_25": NA, "ma_50": NA,
            "rsi":          NA, "high_52w": NA, "low_52w": NA,
            "per":          NA, "pbr": NA,
            "q_score":      NA, "q1": NA, "q3": NA,
            "v_score":      NA, "v1": NA, "v2": NA, "v3": NA,
            "t_score":      NA, "qvt_score": NA,
            "timing_label": NA, "signal_text": NA,
            "roe":               _safe(fund.get("roe")),
            "roa":               _safe(fund.get("roa")),
            "equity_ratio":      _safe(fund.get("equity_ratio")),
            "operating_margin":  _safe(fund.get("operating_margin")),
            "de_ratio":          _safe(fund.get("de_ratio")),
            "interest_coverage": _safe(fund.get("interest_coverage")),
            "ev_ebitda":         _safe(fund.get("ev_ebitda")),
            "industry":          fund.get("industry", NA),
            "sector":            fund.get("sector",   NA),
        }

        if df_raw is None or close_col is None:
            err_cnt += 1
        else:
            try:
                result = calc_scores_for_date(df_raw.copy(), close_col, date_str, fund)
            except Exception:
                result = None

            if result is None:
                skip_cnt += 1   # 休場・データなし
            else:
                row.update({
                    "close":        _safe(result["close"], 0),
                    "ma_25":        _safe(result["ma_25"], 0),
                    "ma_50":        _safe(result["ma_50"], 0),
                    "rsi":          _safe(result["rsi"], 1),
                    "high_52w":     _safe(result["high_52w"], 0),
                    "low_52w":      _safe(result["low_52w"], 0),
                    "per":          _safe(result["per"], 1),
                    "pbr":          _safe(result["pbr"], 2),
                    "q_score":      _safe(result["q_score"], 1),
                    "q1":           _safe(result["q1"], 1),
                    "q3":           _safe(result["q3"], 1),
                    "v_score":      _safe(result["v_score"], 1),
                    "v1":           _safe(result["v1"], 1),
                    "v2":           _safe(result["v2"], 1),
                    "v3":           _safe(result["v3"], 1),
                    "t_score":      _safe(result["t_score"], 1),
                    "qvt_score":    _safe(result["qvt_score"], 1),
                    "timing_label": result.get("timing_label", NA) or NA,
                    "signal_text":  result.get("signal_text",  NA) or NA,
                })
                ok_cnt += 1

        rows.append(row)

    # 銘柄ごとに全日付分を1ファイルに書き出し（UTF-8 BOM付き）
    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"  [{ti:2d}/{len(TICKERS)}] {ticker}: ✅{ok_cnt}日 / ➖{skip_cnt}日(休場) / ❌{err_cnt}日(取得失敗) → {csv_name}")
    summary_log.append((ticker, ok_cnt, skip_cnt, err_cnt))

ok_total   = sum(o for _, o, _, _ in summary_log)
skip_total = sum(s for _, _, s, _ in summary_log)
err_total  = sum(e for _, _, _, e in summary_log)
print(f"\n🏁 完了！ ✅{ok_total}件 / ➖{skip_total}件(休場等) / ❌{err_total}件(取得失敗)")

[1/3] ファンダメンタル取得中（四半期×銘柄）...
      ※yfinanceは現在値のみ提供のため、四半期ごとに再フェッチして
        バックテスト期間中の変化を近似的に反映します

  ▶ 2023-Q4
    [  1/20] 7203.T ... Toyota Motor Corporation
    [  2/20] 3864.T ... Mitsubishi Paper Mills Limited
    [  3/20] 8113.T ... Unicharm Corporation
    [  4/20] 7932.T ... Nippi,Incorporated
    [  5/20] 5333.T ... NGK Insulators, Ltd.

  ▶ 2024-Q1
    [  6/20] 7203.T ... Toyota Motor Corporation
    [  7/20] 3864.T ... Mitsubishi Paper Mills Limited
    [  8/20] 8113.T ... Unicharm Corporation
    [  9/20] 7932.T ... Nippi,Incorporated
    [ 10/20] 5333.T ... NGK Insulators, Ltd.

  ▶ 2024-Q4
    [ 11/20] 7203.T ... Toyota Motor Corporation
    [ 12/20] 3864.T ... Mitsubishi Paper Mills Limited
    [ 13/20] 8113.T ... Unicharm Corporation
    [ 14/20] 7932.T ... Nippi,Incorporated
    [ 15/20] 5333.T ... NGK Insulators, Ltd.

  ▶ 2025-Q1
    [ 16/20] 7203.T ... Toyota Motor Corporation
    [ 17/20] 3864.T ... Mitsubishi Paper Mills Limited
    [ 18/20] 8113.T ... Unicharm

In [9]:
# ═══════════════════════════════════════════════════════════════
# セル6: 出力確認
# ═══════════════════════════════════════════════════════════════

from IPython.display import display
import pandas as pd
from pathlib import Path

csv_files = sorted(Path(OUTPUT_DIR).glob('*.csv'))
print(f'出力ファイル数: {len(csv_files)}')
for f in csv_files:
    df_tmp = pd.read_csv(f, encoding='utf-8-sig')
    print(f'  {f.name}: {len(df_tmp)}行')

if csv_files:
    latest_csv = csv_files[-1]
    df_preview = pd.read_csv(latest_csv, encoding='utf-8-sig')
    print(f'\n--- {latest_csv.name} プレビュー ---')
    show_cols = [c for c in [
        'date', 'quarter', 'ticker', 'company_name',
        'close', 'rsi',
        'q_score', 'v_score', 't_score', 'qvt_score',
        'signal_text',
    ] if c in df_preview.columns]
    display(df_preview[show_cols])


出力ファイル数: 7
  3864T.csv: 104行
  5333T.csv: 104行
  7203T.csv: 104行
  7932T.csv: 104行
  8113T.csv: 104行
  8306T.csv: 42行
  fundamentals_cache.csv: 46行

--- fundamentals_cache.csv プレビュー ---


,ticker
0,3864.T
1,3864.T
2,3864.T
3,3864.T
4,3864.T
5,3864.T
6,3864.T
7,3864.T
8,5333.T
9,5333.T
